# E-Commerce Order Fulfillment Analytics
## Week 2: Data Audit & Cleaning

**Internship Track:** Logistics  
**Project:** E-Commerce Order Fulfillment Analytics  
**Dataset:** E-Commerce Order Fulfillment Dataset — 50K Records

### Objective
Prepare a reliable, analysis-ready dataset by auditing data quality, handling missing values and duplicates, standardizing categorical fields, validating dates, checking `Delivery_Days`, and detecting invalid numeric values and outliers.

> This notebook is for **Week 2 Data Audit & Cleaning**. EDA and visualization are planned for Week 3.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

print("Libraries imported successfully.")

## 1. Load Raw Dataset

Expected path:

`data/E-Commerce_Order_Fulfillment_Dataset_50K_Records.csv`

In [ ]:
DATA_PATH = "../data/E-Commerce_Order_Fulfillment_Dataset_50K_Records.csv"

df = pd.read_csv(DATA_PATH)

raw_record_count = len(df)

print("Raw record count:", raw_record_count)
print("Dataset shape:", df.shape)
display(df.head())

## 2. Initial Data Audit

In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes)

print("\nDataset information:")
df.info()

print("\nSummary statistics:")
display(df.describe(include="all").T)

## 3. Missing-Value Audit

In [ ]:
missing_before = df.isnull().sum()
missing_summary = pd.DataFrame({
    "Missing_Values": missing_before,
    "Missing_Percentage": (missing_before / len(df) * 100).round(2)
}).sort_values("Missing_Values", ascending=False)

display(missing_summary)

total_missing_before = int(missing_before.sum())
print("Total missing values identified:", total_missing_before)

In [ ]:
# Handle missing values
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()

for col in categorical_cols:
    df[col] = df[col].fillna("Unknown")

if "Shipping_Cost" in df.columns:
    df["Shipping_Cost"] = df["Shipping_Cost"].fillna(df["Shipping_Cost"].median())

# Numeric/date columns that cannot be safely imputed are retained for validation
missing_after = df.isnull().sum()
total_missing_after = int(missing_after.sum())

print("Total missing values after handling:", total_missing_after)
display(pd.DataFrame({
    "Before": missing_before,
    "After": missing_after
}).query("Before > 0"))

## 4. Duplicate Audit

In [ ]:
duplicate_rows = int(df.duplicated().sum())
print("Fully duplicated rows:", duplicate_rows)

duplicate_order_ids = 0
if "Order_ID" in df.columns:
    duplicate_order_ids = int(df["Order_ID"].duplicated(keep=False).sum())
    print("Rows belonging to duplicated Order_IDs:", duplicate_order_ids)

In [ ]:
# Remove fully duplicated rows first
before_duplicates = len(df)
df = df.drop_duplicates()
fully_duplicate_rows_removed = before_duplicates - len(df)

# Remove duplicate Order_IDs, keeping the first occurrence
order_id_duplicates_removed = 0
if "Order_ID" in df.columns:
    before_order_id_dedup = len(df)
    df = df.drop_duplicates(subset="Order_ID", keep="first")
    order_id_duplicates_removed = before_order_id_dedup - len(df)

duplicate_rows_removed = fully_duplicate_rows_removed + order_id_duplicates_removed

print("Fully duplicated rows removed:", fully_duplicate_rows_removed)
print("Duplicate Order_ID rows removed:", order_id_duplicates_removed)
print("Total duplicate rows removed:", duplicate_rows_removed)
print("Rows remaining:", len(df))

## 5. Standardize Categorical Values

In [ ]:
categorical_standardization_cols = [
    col for col in [
        "Customer_Region",
        "Product_Category",
        "Shipping_Mode",
        "Delivery_Status"
    ] if col in df.columns
]

for col in categorical_standardization_cols:
    df[col] = df[col].astype("string").str.strip().str.replace(r"\s+", " ", regex=True)
    df[col] = df[col].str.title()

for col in categorical_standardization_cols:
    print(f"\n{col} unique values:")
    print(sorted(df[col].dropna().unique().tolist()))

## 6. Date Validation

In [ ]:
date_cols = ["Order_Date", "Ship_Date", "Delivery_Date"]

for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

print(df[date_cols].dtypes)

invalid_date_mask = pd.Series(False, index=df.index)

if all(col in df.columns for col in date_cols):
    invalid_date_mask = (
        (df["Ship_Date"] < df["Order_Date"]) |
        (df["Delivery_Date"] < df["Ship_Date"]) |
        (df["Delivery_Date"] < df["Order_Date"])
    )

illogical_date_rows = int(invalid_date_mask.sum())
print("Rows with illogical date sequence:", illogical_date_rows)

In [ ]:
# Remove rows with illogical date sequences
df = df.loc[~invalid_date_mask].copy()

print("Rows removed due to illogical dates:", illogical_date_rows)
print("Rows remaining:", len(df))

## 7. Recalculate and Validate Delivery Days

In [ ]:
if all(col in df.columns for col in ["Ship_Date", "Delivery_Date"]):
    df["Calculated_Delivery_Days"] = (
        df["Delivery_Date"] - df["Ship_Date"]
    ).dt.days

    if "Delivery_Days" in df.columns:
        df["Delivery_Days"] = pd.to_numeric(df["Delivery_Days"], errors="coerce")
        df["Delivery_Days_Difference"] = (
            df["Delivery_Days"] - df["Calculated_Delivery_Days"]
        )

        mismatch_count = int(
            df["Delivery_Days_Difference"].fillna(0).ne(0).sum()
        )

        print("Delivery_Days mismatches:", mismatch_count)
        display(
            df.loc[df["Delivery_Days_Difference"] != 0,
                   ["Ship_Date", "Delivery_Date", "Delivery_Days",
                    "Calculated_Delivery_Days", "Delivery_Days_Difference"]]
            .head(10)
        )

## 8. Invalid Negative Values

In [ ]:
# Convert numeric fields
for col in ["Shipping_Cost", "Delivery_Days"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

negative_cost = (
    (df["Shipping_Cost"] < 0).sum()
    if "Shipping_Cost" in df.columns else 0
)

negative_delivery_days = (
    (df["Delivery_Days"] < 0).sum()
    if "Delivery_Days" in df.columns else 0
)

invalid_negative_rows = int(
    ((df["Shipping_Cost"] < 0) | (df["Delivery_Days"] < 0)).sum()
)

print("Negative Shipping_Cost values:", int(negative_cost))
print("Negative Delivery_Days values:", int(negative_delivery_days))
print("Rows with invalid negative values:", invalid_negative_rows)

In [ ]:
negative_mask = pd.Series(False, index=df.index)

if "Shipping_Cost" in df.columns:
    negative_mask = negative_mask | (df["Shipping_Cost"] < 0)

if "Delivery_Days" in df.columns:
    negative_mask = negative_mask | (df["Delivery_Days"] < 0)

df = df.loc[~negative_mask].copy()

print("Rows removed for invalid negative values:", invalid_negative_rows)
print("Rows remaining:", len(df))

## 9. Outlier Detection Using IQR

In [ ]:
def iqr_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper), lower, upper

outlier_results = {}

for col in ["Shipping_Cost", "Delivery_Days"]:
    if col in df.columns:
        mask, lower, upper = iqr_outlier_mask(df[col].dropna())
        full_mask = pd.Series(False, index=df.index)
        full_mask.loc[df[col].dropna().index] = mask
        outlier_results[col] = {
            "count": int(full_mask.sum()),
            "lower_bound": lower,
            "upper_bound": upper
        }

display(pd.DataFrame(outlier_results).T)

### Outlier Treatment Decision

Outliers are **flagged rather than automatically removed** because unusually high shipping costs or delivery times may represent genuine operational cases. Only invalid negative values are removed.

This keeps potentially meaningful business observations in the analysis-ready dataset.

## 10. Final Data Quality Check

In [ ]:
final_missing = int(df.isnull().sum().sum())
final_duplicates = int(df.duplicated().sum())

print("Final record count:", len(df))
print("Final missing values:", final_missing)
print("Final duplicate rows:", final_duplicates)

display(df.head())
display(df.describe().T)

## 11. Findings Summary

In [ ]:
outliers_shipping = outlier_results.get("Shipping_Cost", {}).get("count", 0)
outliers_delivery = outlier_results.get("Delivery_Days", {}).get("count", 0)

findings_summary = pd.DataFrame({
    "Metric": [
        "Raw record count",
        "Missing values identified",
        "Missing values remaining after handling",
        "Duplicate rows removed",
        "Rows removed (illogical date sequence)",
        "Rows removed (invalid negative values)",
        "Outliers flagged (Shipping_Cost)",
        "Outliers flagged (Delivery_Days)",
        "Final clean record count"
    ],
    "Value": [
        raw_record_count,
        total_missing_before,
        total_missing_after,
        duplicate_rows_removed,
        illogical_date_rows,
        invalid_negative_rows,
        outliers_shipping,
        outliers_delivery,
        len(df)
    ]
})

display(findings_summary)

## 12. Export Clean Dataset

The cleaned dataset is exported for use in Week 3 EDA and visualization.

In [ ]:
OUTPUT_PATH = "../data/orders_clean.csv"

# Remove helper columns before export
helper_cols = [
    "Calculated_Delivery_Days",
    "Delivery_Days_Difference"
]

export_df = df.drop(columns=[c for c in helper_cols if c in df.columns])

export_df.to_csv(OUTPUT_PATH, index=False)

print("Clean dataset exported successfully:")
print(OUTPUT_PATH)
print("Final shape:", export_df.shape)

## 13. Conclusion

The Week 2 data audit and cleaning workflow systematically reviewed the raw e-commerce order fulfillment dataset for missing values, duplicate records, categorical inconsistencies, invalid date sequences, inconsistencies in `Delivery_Days`, negative numeric values, and statistical outliers.

The resulting `orders_clean.csv` provides an analysis-ready dataset for **Week 3: Exploratory Data Analysis & Visualization**.

## 14. Next Step — Week 3

Use `data/orders_clean.csv` for:

- Delivery performance analysis
- Regional analysis
- Shipping-mode analysis
- Product-category analysis
- Shipping-cost distributions
- Delivery-status analysis
- Monthly/time-based trends
- Business insights and visualizations

**Week 3 notebook:** `03_eda_visualization.ipynb`